# Lineage Workbench - M Query Column Lineage

This notebook parses Power Query M text from `t_dataset_partitions` and writes column transformation steps to lakehouse Delta tables:
- `t_dataset_column_lineage`
- `t_column_lineage` (alias for compatibility)

It is standalone and does not modify the unified extraction notebook.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
from pyspark.sql.types import StructField, StructType, StringType

if 'targetLakehouseId' not in globals():
    targetLakehouseId = ''
if 'targetLakehouseWorkspaceId' not in globals():
    targetLakehouseWorkspaceId = ''

LAKEHOUSE_ID = str(targetLakehouseId).strip()

def _default_workspace_id():
    for key in ['workspaceId', 'workspaceID']:
        value = globals().get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ''

TARGET_LAKEHOUSE_WORKSPACE_ID = str(targetLakehouseWorkspaceId).strip() if str(targetLakehouseWorkspaceId).strip() else _default_workspace_id()

def _build_target_table_path(table_name: str) -> Optional[str]:
    if not LAKEHOUSE_ID or not TARGET_LAKEHOUSE_WORKSPACE_ID:
        return None
    try:
        default_fs = notebookutils.conf.get('fs.defaultFS')
    except Exception:
        return None
    if not default_fs or '@' not in default_fs:
        return None
    endpoint = default_fs.split('@', 1)[1].rstrip('/')
    return f"abfss://{TARGET_LAKEHOUSE_WORKSPACE_ID}@{endpoint}/{LAKEHOUSE_ID}/Tables/{table_name}"

def _delta_path_exists(path: Optional[str]) -> bool:
    if not path:
        return False
    try:
        return bool(notebookutils.fs.exists(path))
    except Exception:
        return False

def _normalize_for_write(df: pd.DataFrame, required_columns: List[str]) -> pd.DataFrame:
    if df is None:
        df = pd.DataFrame(columns=required_columns)
    result = df.copy()
    for col in required_columns:
        if col not in result.columns:
            result[col] = None
    result = result[required_columns]
    for col in required_columns:
        result[col] = result[col].map(lambda v: None if pd.isna(v) else str(v))
    return result

def clear_table(table_name: str) -> None:
    table_path = _build_target_table_path(table_name)
    try:
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")
    except Exception as ex:
        print(f"Warning: Could not drop table {table_name}: {ex}")

    if table_path and _delta_path_exists(table_path):
        try:
            notebookutils.fs.rm(table_path, True)
        except Exception as ex:
            msg = str(ex).lower()
            if 'pathnotfoundexception' in msg or 'not found' in msg:
                pass
            else:
                print(f"Warning: Could not remove Delta path for {table_name}: {ex}")

def write_table(df: pd.DataFrame, table_name: str, required_columns: List[str]) -> None:
    normalized = _normalize_for_write(df, required_columns)
    if normalized.empty and len(normalized.columns) == 0:
        clear_table(table_name)
        print(f"Created {table_name} with 0 rows")
        return

    schema = StructType([StructField(c, StringType(), True) for c in normalized.columns])
    spark_df = spark.createDataFrame(normalized, schema=schema)
    table_path = _build_target_table_path(table_name)

    if table_path:
        spark_df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').save(table_path)
        try:
            spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")
            spark.sql(f"CREATE TABLE `{table_name}` USING DELTA LOCATION '{table_path}'")
            spark.sql(f"REFRESH TABLE `{table_name}`")
        except Exception as ex:
            print(f"Warning: Could not register SQL table {table_name}: {ex}")
    else:
        spark_df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)

    sql_count = None
    try:
        sql_count = spark.table(table_name).count()
    except Exception:
        sql_count = None

    if sql_count is not None:
        print(f"Created {table_name} with {len(normalized)} rows (SQL count={sql_count})")
    else:
        print(f"Created {table_name} with {len(normalized)} rows")

In [ ]:
@dataclass
class MQueryStep:
    step_name: str
    step_order: int
    transformation_function: str
    step_expression: str
    affected_columns: Set[str] = field(default_factory=set)
    renamed_columns: Dict[str, str] = field(default_factory=dict)
    added_columns: Set[str] = field(default_factory=set)
    removed_columns: Set[str] = field(default_factory=set)
    affects_entire_table: bool = False


class MQueryParser:
    TABLE_WIDE_FUNCTIONS = {
        'Table.SelectRows', 'Table.Sort', 'Table.Distinct', 'Table.FirstN',
        'Table.LastN', 'Table.Skip', 'Table.RemoveRowsWithErrors', 'Table.Buffer',
        'Sql.Database', 'Sql.Databases', 'Table.FromRecords', 'Excel.Workbook',
        'Csv.Document', 'Json.Document', 'Table.NestedJoin', 'Table.Join',
    }

    @staticmethod
    def _strip_comments(m_query: str) -> str:
        text = re.sub(r'//.*?$', '', m_query, flags=re.MULTILINE)
        text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)
        return text

    @staticmethod
    def _normalize_step_name(raw: str) -> str:
        value = raw.strip()
        if value.startswith('#"') and value.endswith('"'):
            return value[2:-1]
        if value.startswith('"') and value.endswith('"'):
            return value[1:-1]
        return value

    @staticmethod
    def _split_top_level(text: str, delimiter: str = ',') -> List[str]:
        parts: List[str] = []
        buf: List[str] = []
        paren = brace = bracket = 0
        in_quote = False
        i = 0

        while i < len(text):
            ch = text[i]

            if ch == '"':
                if i + 1 < len(text) and text[i + 1] == '"':
                    buf.append(ch)
                    buf.append(text[i + 1])
                    i += 2
                    continue
                in_quote = not in_quote
                buf.append(ch)
                i += 1
                continue

            if not in_quote:
                if ch == '(':
                    paren += 1
                elif ch == ')':
                    paren = max(0, paren - 1)
                elif ch == '{':
                    brace += 1
                elif ch == '}':
                    brace = max(0, brace - 1)
                elif ch == '[':
                    bracket += 1
                elif ch == ']':
                    bracket = max(0, bracket - 1)
                elif ch == delimiter and paren == 0 and brace == 0 and bracket == 0:
                    parts.append(''.join(buf).strip())
                    buf = []
                    i += 1
                    continue

            buf.append(ch)
            i += 1

        tail = ''.join(buf).strip()
        if tail:
            parts.append(tail)
        return parts

    def parse_query(self, m_query: str) -> List[MQueryStep]:
        text = self._strip_comments(m_query or '')
        let_match = re.search(r'\blet\b(.*)\bin\b', text, flags=re.IGNORECASE | re.DOTALL)
        if not let_match:
            return []

        let_body = let_match.group(1).strip()
        assignments = self._split_top_level(let_body, ',')
        steps: List[MQueryStep] = []

        for idx, assignment in enumerate(assignments):
            if '=' not in assignment:
                continue
            left, right = assignment.split('=', 1)
            step_name = self._normalize_step_name(left)
            expression = right.strip()
            fn_match = re.match(r'([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)*)\s*\(', expression)
            fn_name = fn_match.group(1) if fn_match else 'unknown'

            step = MQueryStep(
                step_name=step_name or f'Step {idx + 1}',
                step_order=idx + 1,
                transformation_function=fn_name,
                step_expression=expression,
            )

            self._analyze_step(step)
            steps.append(step)

        return steps

    def _analyze_step(self, step: MQueryStep) -> None:
        func = step.transformation_function
        expr = step.step_expression

        if func in self.TABLE_WIDE_FUNCTIONS:
            step.affects_entire_table = True
            return

        if func == 'Table.RenameColumns':
            for old_name, new_name in re.findall(r'\{\s*\"([^\"]+)\"\s*,\s*\"([^\"]+)\"\s*\}', expr):
                step.renamed_columns[old_name] = new_name
                step.affected_columns.update([old_name, new_name])
            return

        if func == 'Table.RemoveColumns':
            cols = re.findall(r'\"([^\"]+)\"', expr)
            step.removed_columns.update(cols)
            step.affected_columns.update(cols)
            return

        if func == 'Table.SelectColumns':
            cols = re.findall(r'\"([^\"]+)\"', expr)
            step.affected_columns.update(cols)
            return

        if func == 'Table.AddColumn':
            m = re.search(r'Table\.AddColumn\s*\([^,]+,\s*\"([^\"]+)\"', expr)
            if m:
                col = m.group(1)
                step.added_columns.add(col)
                step.affected_columns.add(col)
            return

        if func == 'Table.ReplaceValue':
            m = re.search(r'Replacer\.\w+\s*,\s*\{([^}]*)\}', expr)
            if m:
                cols = re.findall(r'\"([^\"]+)\"', m.group(1))
                step.affected_columns.update(cols)
            return

        if func == 'Table.TransformColumns':
            cols = re.findall(r'\{\s*\"([^\"]+)\"', expr)
            step.affected_columns.update(cols)
            return

        if func in ('Table.ExpandRecordColumn', 'Table.ExpandTableColumn'):
            m = re.search(r'Table\.Expand\w+\s*\([^,]+,\s*\"([^\"]+)\"\s*,\s*\{([^}]*)\}', expr)
            if m:
                source_col = m.group(1)
                new_cols = re.findall(r'\"([^\"]+)\"', m.group(2))
                step.affected_columns.add(source_col)
                step.added_columns.update(new_cols)
                step.affected_columns.update(new_cols)
            return

        if func == 'Table.DuplicateColumn':
            m = re.search(r'Table\.DuplicateColumn\s*\([^,]+,\s*\"([^\"]+)\"\s*,\s*\"([^\"]+)\"', expr)
            if m:
                src_col, new_col = m.group(1), m.group(2)
                step.affected_columns.add(src_col)
                step.added_columns.add(new_col)
                step.affected_columns.add(new_col)
            return

        if func == 'Table.SplitColumn':
            m = re.search(r'Table\.SplitColumn\s*\([^,]+,\s*\"([^\"]+)\"', expr)
            if m:
                src_col = m.group(1)
                step.affected_columns.add(src_col)
                step.removed_columns.add(src_col)
            return

        # Generic fallback: if function is unknown, keep table-wide off.

    def trace_column_lineage(self, steps: List[MQueryStep], final_column_name: str) -> List[Tuple[str, MQueryStep]]:
        lineage: List[Tuple[str, MQueryStep]] = []
        current_name = final_column_name

        for step in reversed(steps):
            matched = False

            if current_name in step.renamed_columns.values():
                old_name = next((k for k, v in step.renamed_columns.items() if v == current_name), current_name)
                lineage.append((current_name, step))
                current_name = old_name
                matched = True
            elif current_name in step.renamed_columns:
                lineage.append((current_name, step))
                matched = True
            elif current_name in step.affected_columns or step.affects_entire_table:
                lineage.append((current_name, step))
                matched = True

            if current_name in step.added_columns:
                break

            if not matched and step.affects_entire_table:
                lineage.append((current_name, step))

        return lineage


In [ ]:
def _to_pandas(table_name: str) -> pd.DataFrame:
    try:
        return spark.table(table_name).toPandas()
    except Exception as ex:
        print(f"Warning: Could not read {table_name}: {ex}")
        return pd.DataFrame()

def _sanitize_cols(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or (df.empty and len(df.columns) == 0):
        return pd.DataFrame()
    out = df.copy()
    out.columns = out.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
    return out

def _pick_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _json_safe(value):
    if pd.isna(value):
        return None
    return str(value)

def _df_to_records(df: pd.DataFrame) -> List[Dict[str, Optional[str]]]:
    if df is None or df.empty:
        return []
    return [
        {col: _json_safe(row.get(col)) for col in df.columns}
        for _, row in df.iterrows()
    ]

def _read_first_non_empty_df(table_names: List[str]) -> Tuple[pd.DataFrame, Optional[str]]:
    for table_name in table_names:
        df = _sanitize_cols(_to_pandas(table_name))
        if not df.empty:
            return df, table_name
    return pd.DataFrame(), None

def sync_snapshot_dimensions(
    column_lineage_df: pd.DataFrame,
    dataset_columns_df: pd.DataFrame,
    partitions_df: pd.DataFrame,
    snapshot_path: str = 'Files/lineage/snapshots/graph_snapshot.json'
) -> None:
    snapshot = {
        'nodes': [],
        'edges': [],
        'dimensions': {},
    }

    try:
        if notebookutils.fs.exists(snapshot_path):
            raw = notebookutils.fs.head(snapshot_path, 10_000_000)
            loaded = json.loads(raw) if raw and str(raw).strip() else {}
            if isinstance(loaded, dict):
                snapshot = loaded
    except Exception as ex:
        print(f"Warning: Could not read existing graph snapshot: {ex}")

    if not isinstance(snapshot.get('nodes'), list):
        snapshot['nodes'] = []
    if not isinstance(snapshot.get('edges'), list):
        snapshot['edges'] = []

    dimensions = snapshot.get('dimensions')
    if not isinstance(dimensions, dict):
        dimensions = {}
        snapshot['dimensions'] = dimensions

    # Always refresh query-step lineage, columns, and partitions for detail view reliability.
    dimensions['columnLineage'] = _df_to_records(column_lineage_df)
    dimensions['columns'] = _df_to_records(dataset_columns_df)
    dimensions['partitions'] = _df_to_records(partitions_df)

    # Load only essential supplemental dimensions to avoid large memory spikes.
    dimension_sources = {
        'measures': ['t_dataset_measures', 't_dataset_measure'],
        'tables': ['t_dataset_tables'],
        'relationships': ['t_dataset_relations', 't_dataset_relationships'],
        'semanticModels': ['t_dataset_semantic_models', 't_datamodel_semantic_models', 'lineage_semantic_models'],
    }

    for dim_key, table_candidates in dimension_sources.items():
        try:
            df, source_table = _read_first_non_empty_df(table_candidates)
            if not df.empty:
                records = _df_to_records(df)
                dimensions[dim_key] = records
                print(f"Snapshot dimension '{dim_key}' loaded from {source_table} ({len(records)} row(s)).")
            elif dim_key not in dimensions:
                dimensions[dim_key] = []
        except Exception as ex:
            print(f"Warning: Could not hydrate snapshot dimension '{dim_key}': {ex}")
            if dim_key not in dimensions:
                dimensions[dim_key] = []

    notebookutils.fs.put(snapshot_path, json.dumps(snapshot, ensure_ascii=False, indent=2), True)
    print(
        f"Updated {snapshot_path}: "
        f"columnLineage={len(dimensions['columnLineage'])}, "
        f"columns={len(dimensions['columns'])}, "
        f"partitions={len(dimensions['partitions'])}"
    )

partitions_raw = _to_pandas('t_dataset_partitions')
dataset_columns_raw = _to_pandas('t_dataset_columns')

partitions = _sanitize_cols(partitions_raw)
dataset_columns = _sanitize_cols(dataset_columns_raw)

if partitions.empty:
    print('No rows found in t_dataset_partitions. Writing empty lineage tables.')

required_output_columns = [
    'dataset_id',
    'workspace_id',
    'power_bi_table_name',
    'partition_name',
    'final_column_name',
    'column_name_at_step',
    'step_order',
    'step_name',
    'transformation_function',
    'step_expression',
    'affects_entire_table',
    'operation_type',
    'created_at_utc',
]

if partitions.empty:
    empty_df = pd.DataFrame(columns=required_output_columns)
    write_table(empty_df, 't_dataset_column_lineage', required_output_columns)
    write_table(empty_df, 't_column_lineage', required_output_columns)
    sync_snapshot_dimensions(empty_df, dataset_columns, partitions)
else:
    dataset_id_col = _pick_column(partitions, ['dataset_id'])
    workspace_id_col = _pick_column(partitions, ['workspace_id'])
    partition_name_col = _pick_column(partitions, ['partition_name'])
    query_col = _pick_column(partitions, ['query', 'source'])
    table_name_col = _pick_column(partitions, ['table_name', 'name'])

    if not dataset_id_col or not query_col:
        raise ValueError('t_dataset_partitions must contain dataset_id and query (or source) columns.')

    # Build lookup: (dataset_id, table_name) -> set(column_name)
    dataset_columns_lookup: Dict[Tuple[str, str], Set[str]] = {}
    if not dataset_columns.empty:
        d_dataset_col = _pick_column(dataset_columns, ['dataset_id'])
        d_table_col = _pick_column(dataset_columns, ['table_name'])
        d_column_col = _pick_column(dataset_columns, ['column_name'])

        if d_dataset_col and d_table_col and d_column_col:
            for _, row in dataset_columns.iterrows():
                did = str(row.get(d_dataset_col) or '').strip()
                tname = str(row.get(d_table_col) or '').strip()
                cname = str(row.get(d_column_col) or '').strip()
                if did and tname and cname:
                    key = (did.lower(), tname.lower())
                    if key not in dataset_columns_lookup:
                        dataset_columns_lookup[key] = set()
                    dataset_columns_lookup[key].add(cname)

    parser = MQueryParser()
    rows: List[Dict[str, str]] = []
    now_utc = datetime.now(timezone.utc).isoformat()

    for _, part in partitions.iterrows():
        dataset_id = str(part.get(dataset_id_col) or '').strip()
        if not dataset_id:
            continue

        workspace_id = str(part.get(workspace_id_col) or '').strip() if workspace_id_col else ''
        partition_name = str(part.get(partition_name_col) or '').strip() if partition_name_col else ''
        power_bi_table_name = str(part.get(table_name_col) or '').strip() if table_name_col else ''
        query_text = str(part.get(query_col) or '').strip()

        if not query_text:
            continue

        steps = parser.parse_query(query_text)
        if not steps:
            continue

        target_columns: Set[str] = set()
        if power_bi_table_name:
            key = (dataset_id.lower(), power_bi_table_name.lower())
            target_columns = set(dataset_columns_lookup.get(key, set()))

        if not target_columns:
            touched = set()
            for s in steps:
                touched.update(s.affected_columns)
                touched.update(s.added_columns)
                touched.update(s.renamed_columns.values())
            target_columns = {c for c in touched if c}

        for final_col in sorted(target_columns):
            lineage = parser.trace_column_lineage(steps, final_col)
            if not lineage:
                continue

            for col_at_step, step in lineage:
                operation_type = 'inherited'
                if col_at_step in step.renamed_columns.values():
                    operation_type = 'rename_to'
                elif col_at_step in step.renamed_columns:
                    operation_type = 'rename_from'
                elif col_at_step in step.added_columns:
                    operation_type = 'created'
                elif col_at_step in step.removed_columns:
                    operation_type = 'removed'
                elif step.affects_entire_table:
                    operation_type = 'table_operation'
                elif col_at_step in step.affected_columns:
                    operation_type = 'transformed'

                rows.append({
                    'dataset_id': dataset_id,
                    'workspace_id': workspace_id,
                    'power_bi_table_name': power_bi_table_name,
                    'partition_name': partition_name,
                    'final_column_name': final_col,
                    'column_name_at_step': col_at_step,
                    'step_order': str(step.step_order),
                    'step_name': step.step_name,
                    'transformation_function': step.transformation_function,
                    'step_expression': step.step_expression,
                    'affects_entire_table': 'true' if step.affects_entire_table else 'false',
                    'operation_type': operation_type,
                    'created_at_utc': now_utc,
                })

    out_df = pd.DataFrame(rows, columns=required_output_columns)
    out_df = out_df.drop_duplicates()

    write_table(out_df, 't_dataset_column_lineage', required_output_columns)
    write_table(out_df, 't_column_lineage', required_output_columns)
    sync_snapshot_dimensions(out_df, dataset_columns, partitions)

    print(f"Column lineage rows generated: {len(out_df)}")
    if len(out_df) > 0:
        print('Sample output:')
        print(out_df.head(10).to_string(index=False))

In [ ]:
print('M Query column lineage notebook completed successfully.')